#### Environment Setup & Raw Data Ingestion

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("Loading datasets...")
# Make sure these paths are perfectly correct for your computer
all_data_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx"
breakdown_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx"

df_all = pd.read_excel(all_data_file)
df_breakdown = pd.read_excel(breakdown_file)

# Handle column spelling differences
col_healthy = 'Breakdown' if 'Breakdown' in df_all.columns else 'Brekadown'
col_broken = 'Breakdown' if 'Breakdown' in df_breakdown.columns else 'Brekadown'

# Isolate the healthy data 
df_healthy = df_all[df_all[col_healthy].isna()].copy()
print(f"Loaded {len(df_healthy)} healthy records and {len(df_breakdown)} breakdown records.")

Loading datasets...
Loaded 1604 healthy records and 704 breakdown records.


#### Feature Engineering & Data Fusion

In [2]:
print("Combining Vibration, Electrical, and Mechanical Data...")

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# The 10 extra features to track alongside vibration
extra_features = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]

# Process Healthy Data
h_vibs = pd.DataFrame(df_healthy['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
h_extras = df_healthy[extra_features].reset_index(drop=True)
df_h_ml = pd.concat([h_extras, h_vibs], axis=1)
df_h_ml['Label'] = 'Healthy Baseline'

# Process Breakdown Data
b_vibs = pd.DataFrame(df_breakdown['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
b_extras = df_breakdown[extra_features].reset_index(drop=True)
df_b_ml = pd.concat([b_extras, b_vibs], axis=1)
df_b_ml['Label'] = df_breakdown[col_broken].values

# Combine into one Master Dataset
df_ml_master = pd.concat([df_h_ml, df_b_ml]).fillna(0)
print(f"Master Dataset created! The AI will look at {df_ml_master.shape[1] - 1} different variables per machine.")

Combining Vibration, Electrical, and Mechanical Data...
Master Dataset created! The AI will look at 69 different variables per machine.


#### Label Encoding & Feature Standardization

In [3]:
print("Formatting Data for Deep Learning...")

# Separate numbers (X) from labels (Y)
X_raw = df_ml_master.drop(columns=['Label']).values
y_text = df_ml_master['Label'].values

# Encode text labels into numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_text)
y_categorical = to_categorical(y_encoded)

# Scale the data so RPM (4000) and Current (1.5) are treated equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save the Scaler and Encoder to your hard drive
with open('sewing_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('sewing_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
    
print("Translators saved successfully.")

Formatting Data for Deep Learning...
Translators saved successfully.


#### Time-Series Transformation (Sliding Window Generation)

In [4]:
print("Creating Stratified Sliding Time Windows...")

def create_sequences(data_X, data_Y, time_steps=5):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - time_steps):
        X_seq.append(data_X[i : (i + time_steps)])
        y_seq.append(data_Y[i + time_steps]) 
    return np.array(X_seq), np.array(y_seq)

TIME_STEPS = 5 
# Find exactly where the healthy data ends and breakdowns begin
num_healthy_records = len(df_h_ml)

# 1. Split scaled data back into Healthy vs Breakdown groups
X_healthy = X_scaled[:num_healthy_records]
y_healthy = y_categorical[:num_healthy_records]

X_broken = X_scaled[num_healthy_records:]
y_broken = y_categorical[num_healthy_records:]

# 2. Create sliding windows for each group independently
X_seq_h, y_seq_h = create_sequences(X_healthy, y_healthy, TIME_STEPS)
X_seq_b, y_seq_b = create_sequences(X_broken, y_broken, TIME_STEPS)

# 3. Perform chronological train/test split (80/20) on each group
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_seq_h, y_seq_h, test_size=0.2, random_state=42, shuffle=False)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_seq_b, y_seq_b, test_size=0.2, random_state=42, shuffle=False)

# 4. Combine the healthy and broken sets
X_train = np.vstack((X_train_h, X_train_b))
X_test = np.vstack((X_test_h, X_test_b))
y_train = np.vstack((y_train_h, y_train_b))
y_test = np.vstack((y_test_h, y_test_b))

# 5. Shuffle ONLY the training data so batch learning is balanced
np.random.seed(42) 
idx = np.random.permutation(len(X_train))
X_train = X_train[idx]
y_train = y_train[idx]

print(f"Corrected LSTM Train Shape: {X_train.shape}")
print(f"Corrected LSTM Test Shape: {X_test.shape}")

# --- VERIFICATION BLOCK ---
y_train_flat = np.argmax(y_train, axis=1)
unique_classes, counts = np.unique(y_train_flat, return_counts=True)
print("\nVERIFICATION: Distribution of labels in the Training Set:")
for cls, count in zip(unique_classes, counts):
    label_name = encoder.inverse_transform([cls])[0]
    print(f" - {label_name}: {count} sequences")

Creating Stratified Sliding Time Windows...
Corrected LSTM Train Shape: (1838, 5, 69)
Corrected LSTM Test Shape: (460, 5, 69)

VERIFICATION: Distribution of labels in the Training Set:
 - Code Uneven: 50 sequences
 - Cut/Needle Hole: 50 sequences
 - Gathering/Puckering: 50 sequences
 - Healthy Baseline: 1279 sequences
 - High Foot Pressure: 50 sequences
 - Needle Breakages: 45 sequences
 - Oil Mark: 50 sequences
 - Pneumatic Issues: 50 sequences
 - Roping: 50 sequences
 - Skip Stitches/Slip: 50 sequences
 - Thread Breakages: 50 sequences
 - Thread Jamming: 50 sequences
 - Waveness: 14 sequences


#### Deep Learning Architecture Definition

In [5]:
print("Building the Predictive LSTM Brain...")

model = Sequential()

# First Layer: Reads the sliding time window
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))

# Second Layer: Deep pattern extraction
model.add(LSTM(32, return_sequences=False))
model.add(Dropout(0.2))

# Dense processing
model.add(Dense(32, activation='relu'))

# Output Layer: Predicts the exact breakdown category
num_classes = y_categorical.shape[1]
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Building the Predictive LSTM Brain...


c:\Users\deela\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 5, 64)          │        34,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,304 (188.69 KB)

 Trainable params: 48,304 (188.69 KB)

 Non-trainable params: 0 (0.00 B)

#### Model Training, Evaluation, & Export

In [6]:
print("Commencing Model Training...")

history = model.fit(
    X_train, y_train,
    epochs=50,           
    batch_size=16,       
    validation_data=(X_test, y_test), 
    verbose=1
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"FINAL PREDICTIVE TEST ACCURACY: {accuracy * 100:.2f}%")
print("="*40)

model.save("sewing_machine_predictive_lstm.h5")
print("✅ SUCCESS! Saved the .h5 model to your computer.")

Commencing Model Training...
Epoch 1/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5921 - loss: 1.9743 - val_accuracy: 0.7261 - val_loss: 1.1274
Epoch 2/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7391 - loss: 0.8394 - val_accuracy: 0.6957 - val_loss: 1.4166
Epoch 3/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7971 - loss: 0.6467 - val_accuracy: 0.6957 - val_loss: 2.0647
Epoch 4/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8705 - loss: 0.4779 - val_accuracy: 0.6935 - val_loss: 2.1242
Epoch 5/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9173 - loss: 0.2988 - val_accuracy: 0.6978 - val_loss: 2.4019
Epoch 6/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9473 - loss: 0.2104 - val_accuracy: 0.7065 - val_loss: 2.6269
Epoch 7/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9729 - loss: 0.1373 - val_accuracy: 0.7130 - val_loss: 2.6528
Epoch 8/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9743 - lo


FINAL PREDICTIVE TEST ACCURACY: 73.48%
✅ SUCCESS! Saved the .h5 model to your computer.
